# Parameter identifiability screen

For each candidate parameter we sweep it across a range, hold every other parameter fixed
at a baseline, run one `forward_pass()` per sampled value, collapse each simulation to a
few scalar summaries of the survey detected-positive CT signal , and correlate the swept value against each summary (Spearman, robust
to monotone nonlinearity).

- **|r| near 0** across all summaries -> the parameter leaves essentially no fingerprint in
  the CT surveillance signal under the current model, so it is not identifiable from this
  data source (and no amount of simulation budget fixes that -- it would need a structural
  change, e.g. coupling the viral-load curve to infection length).
- **|r| clearly non-zero** -> the parameter carries signal and is a candidate for the
  multi-parameter inference.

`beta` and `t_exposed` are included as positive controls: both already recover well in the main
analysis, so they should land at high |r|.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

# run_design_sweep sets a headless backend on import
import run_design_sweep as rds
%matplotlib inline

from memilio.simulation.abm import ABMPopulation, TestingBudget
from abm_batch import run_batch

In [ ]:
SIM_DAYS, N_CT_BINS = 30, 41

RNG = np.random.default_rng(2025)
POPULATION = ABMPopulation(total_population=100_000)
DESIGN = TestingBudget(event_budget_fraction=0.1, test_period_days=1)

BASELINE = {
    "beta":                       1.0,
    "t_exposed":                  6.0,
    "time_presymptomatic":        2.25,
    "time_asymptomatic_recovery": 8.5,
    "symptom_prob":               0.6,
}

N_SCREEN = 150
CANDIDATES = {
    "beta":                       np.exp(RNG.uniform(np.log(0.5), np.log(2.0), N_SCREEN)),
    "t_exposed":                      RNG.uniform(2.0, 10.0, N_SCREEN),
    "time_presymptomatic":        RNG.uniform(0.5, 5.0, N_SCREEN),
    "time_asymptomatic_recovery": RNG.uniform(4.0, 14.0, N_SCREEN),
    "symptom_prob":               RNG.uniform(0.1, 0.9, N_SCREEN),
}
SUMMARY_KEYS = ["total_positives", "mean_ct", "frac_first_half"]

In [ ]:
def summarize(out):
    """Collapse one forward_pass() dict to scalar summaries of the SURVEY detected-positive
    CT signal (records_to_histogram bins the source-0 positives to a (SIM_DAYS, N_CT_BINS)
    image):
      total_positives  -- overall detected-positive count (prevalence/incidence channel)
      mean_ct          -- mean CT value of detected positives (viral-load/timing channel)
      frac_first_half  -- fraction of positives in the first half of the window (temporal shape)
    """
    per_ct = rds.records_to_histogram(out, sources={rds.SURVEY}, sim_days=SIM_DAYS, n_bins=N_CT_BINS)
    daily_pos = per_ct.sum(axis=1)              # (SIM_DAYS,)
    total_pos = daily_pos.sum()
    ct_totals = per_ct.sum(axis=0)              # (N_CT_BINS,)
    mean_ct = (np.arange(N_CT_BINS) * ct_totals).sum() / ct_totals.sum() if ct_totals.sum() > 0 else np.nan
    first_half = daily_pos[np.arange(SIM_DAYS) < SIM_DAYS / 2].sum()
    frac_first_half = first_half / total_pos if total_pos > 0 else np.nan
    return {"total_positives": total_pos, "mean_ct": mean_ct, "frac_first_half": frac_first_half}


def screen_parameter(name, values, baseline, population, design, max_workers=None):
    """Sweep one parameter across `values` (others fixed at `baseline`), returning the swept
    values alongside each scalar summary as parallel arrays."""
    param_sets = [{**baseline, name: float(v)} for v in values]
    outs = run_batch(param_sets, population, design, max_workers=max_workers)
    summ = [summarize(o) for o in outs]
    out = {"values": np.asarray(values, dtype=float)}
    for key in SUMMARY_KEYS:
        out[key] = np.array([s[key] for s in summ])
    return out

In [ ]:
results = {}
for name, values in CANDIDATES.items():
    print(f"screening {name} ...")
    results[name] = screen_parameter(name, values, BASELINE, POPULATION, DESIGN)
print("done")

In [ ]:
# Spearman correlation of each swept parameter against each summary.
corr = {
    name: {k: spearmanr(results[name]["values"], results[name][k], nan_policy="omit").statistic
           for k in SUMMARY_KEYS}
    for name in CANDIDATES
}
corr_df = pd.DataFrame(corr).T[SUMMARY_KEYS]
print(corr_df.round(3))

# Ranked "does this parameter carry any signal": max |r| across summaries.
absmax = corr_df.abs().max(axis=1).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(absmax.index, absmax.values, color="tab:blue")
ax.axvline(0.3, color="k", linestyle="--", linewidth=1, label="rough weak/strong guide (|r|=0.3)")
ax.set_xlabel("max |Spearman r| across summaries")
ax.set_title("Parameter identifiability screen (model-free)")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Scatter every candidate against all three summaries with Spearman r
fig, axes = plt.subplots(len(SUMMARY_KEYS), len(CANDIDATES),
                         figsize=(3.2 * len(CANDIDATES), 3.0 * len(SUMMARY_KEYS)), squeeze=False)
for row, key in enumerate(SUMMARY_KEYS):
    for col, name in enumerate(CANDIDATES):
        ax = axes[row][col]
        ax.scatter(results[name]["values"], results[name][key], s=8, alpha=0.5)
        ax.set_title(f"r={corr_df.loc[name, key]:.2f}", fontsize=8)
        if col == 0:
            ax.set_ylabel(key)
        if row == len(SUMMARY_KEYS) - 1:
            ax.set_xlabel(name)
fig.tight_layout()
plt.show()

### Interpretation

- Candidates landing at high |r| (alongside the `beta`/`t_exposed` controls) have a real channel
  into the PCR surveillance signal and are the ones worth adding to the joint BayesFlow
  inference next.
- Candidates near zero are not identifiable from this data source under the current model;
  don't spend inference budget on them (or revisit the model structure -- e.g. couple the
  viral-load curve to infection length -- if you believe they *should* be informative).
- `mean_ct` isolates the viral-load/CT channel specifically, while `total_positives` and
  `frac_first_half` capture prevalence and epidemic-timing channels. A parameter that only
  moves the timing/prevalence summaries but not `mean_ct` is informative via the epidemic
  curve rather than via the individual CT distribution.
- This is a *screen*, not the answer: a promising candidate should be confirmed by actually
  adding it to the amortized inference and checking recovery/calibration there.